# PROC GVARCLUS로 생산라인 센서 중복성 줄이기

## 요약

다중 구역 제조 라인은 수십 개의 상관관계가 있는 센서 판독값을 스트리밍하며, 그중 다수가 동일한 근본 신호를 담고 있습니다. 이 노트북은 **PROC GVARCLUS**(변수 군집화)를 사용해 13개 공정 센서를 네 개의 서로 배타적인 군집으로 묶은 다음, 각 군집의 R제곱 값을 읽어 군집당 대표 계측기를 하나씩 선택함으로써 공정 정보 손실 없이 SPC 모니터링 부담을 줄입니다. 네 군집 중 세 개는 물리적 구역과 깔끔하게 대응합니다(평균 R제곱 0.92, 0.93, 0.96). 네 번째는 절차가 분리해 낸 단일 채널 군집으로, 이는 무시하지 않고 별도로 살펴봅니다.

## 데이터 출처

모든 데이터는 `call streaminit(20260531)`과 `rand()`로 인라인 생성됩니다 — 외부나 네트워크 입력 없음.

| 데이터셋 | 행 수 | 주요 변수 | 설명 |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | 3구역 생산 라인의 합성 판독값. 같은 구역 내 센서들은 잠재 구동 신호를 공유하여(높은 상관관계), 구역 간 계측기(온도는 구역 1, 압력은 구역 2, 진동은 구역 3에 연결)를 추가해 현실적인 중복성을 만듭니다. DATA 스텝의 루프는 300 사이클을 요청하지만, 이 평가 환경은 비라이선스 모드로 실행되어 처음 100개 관측치만 유지합니다 — 군집 구조를 깔끔하게 복원하기에 충분한 수준입니다. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | 입력 센서별 행 하나: 배정된 군집과 그 군집 성분에 대한 R제곱. `OUTPUT OUT=` 문으로 생성됩니다. |

# PROC GVARCLUS로 생산라인 센서 중복성 줄이기

계측된 생산 라인에서는 겹치는 물리 현상을 측정하는 수십 개의 센서를 기록하는 것이 흔합니다 — 한 구역에 있는 여러 열전대, 중복된 압력 변환기, 같은 축을 추적하는 진동 프로브 등. 모든 채널을 관리도나 후속 모델에 그대로 투입하면 모니터링 노력이 낭비되고 다중공선성이 커집니다.

**변수 군집화**는 이를 직접 해결합니다. `PROC GVARCLUS`는 수치형 변수를 *서로 배타적인* 군집으로 분할하며, 각 군집은 구성원들의 제1주성분으로 요약됩니다. 함께 움직이는 센서들은 같은 군집에 속하게 되고, 엔지니어는 군집당 대표 채널을 하나씩 유지할 수 있습니다.

이 노트북에서는 다음을 수행합니다:

1. 의도적으로 중복된 센서를 가진 3구역 라인의 판독값을 합성합니다(루프는 300 사이클을 요청하며, 여기서는 100개가 유지됩니다).
2. `MAXCLUSTERS=4`로 `PROC GVARCLUS`를 사용해 13개 채널을 군집화합니다.
3. 군집 배정을 출력 데이터셋에 담아 요약합니다.
4. R제곱 값을 해석하여 지속적인 SPC를 위해 군집당 계측기 하나를 선택합니다.

## 1단계 — 합성 센서 데이터 생성

세 개의 공정 구역을 시뮬레이션하며, 각 구역은 숨겨진 구동 신호(`zone1_a`, `zone2_a`, `zone3_a`)를 가집니다. 나머지 채널은 각 구역 구동 신호의 잡음이 섞인 선형 함수이므로, 구역 내에서는 강한 상관관계를 가지지만 구역 간에는 거의 독립적입니다. 또한 입구/출구 온도를 구역 1에, 두 압력 변환기를 구역 2에 연결하여 실제 라인에서 볼 수 있는 계측기 간 중복성을 모방합니다. 고정된 시드로 실행을 재현 가능하게 합니다. 루프는 300 사이클을 요청하며, 비라이선스 모드에서 엔진은 처음 100개 관측치를 유지하고, 아래 NOTE가 이를 보고합니다.

In [1]:
데이터 process_sensors;
    호출 streaminit(20260531);
    반복 cycle = 1 까지 300;
        /* Zone 1: latent driver plus two redundant probes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latent driver plus two redundant probes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latent driver plus one redundant probe */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Cross-instrument channels tied to existing drivers */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        출력;
    종료;
    제거 cycle;
실행;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.08 seconds
  cpu   0.08 seconds


## 2단계 — 센서 군집화

13개 채널 전체에 `PROC GVARCLUS`를 호출합니다. `VAR` 문은 군집화할 수치형 센서를 나열합니다. `MAXCLUSTERS=4`로 분할을 네 군집으로 제한합니다(물리적 구역당 대략 하나의 군집을 예상하되 약간의 여유를 둡니다). `PLOTS=ALL`을 지정한 `ODS GRAPHICS ON`은 변수 군집 덴드로그램을 활성화합니다. 엔진은 해당 SVG를 인라인 렌더링 대신 작업 디렉터리에 기록하므로, 아래에서 읽는 구조는 출력된 **변수 군집 배정** 표와 군집별 고유값 표에서 나옵니다.

`OUTPUT OUT=` 문은 변수-군집 배정을 각 변수의 자기 군집에 대한 R제곱과 함께 이후 프로그램에서 사용할 수 있는 데이터셋에 기록합니다.

In [2]:
ODS GRAPHICS ON;

처리 gvarclus 데이터=process_sensors maxclusters=4 PLOTS=ALL;
    변수 zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    출력 out=clusters;
실행;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## 3단계 — SUMMARY 옵션으로 재실행

`SUMMARY` 옵션으로 절차를 두 번째 실행하면 동일한 분할이 유지됩니다. 이번 실행에서는 `PLOTS=NONE`으로 그래픽을 끕니다. 이 환경에서는 출력 보고서가 2단계와 동일합니다 — 동일한 13행 배정 표, 동일한 네 군집, 동일한 고유값 요약 — 따라서 이 셀은 주로 `SUMMARY`와 `PLOTS=NONE` 옵션이 파싱되고 실행됨을 보여주는 용도이며, 새로운 수치를 추가할 것으로 기대하지 않습니다.

In [3]:
처리 gvarclus 데이터=process_sensors maxclusters=4 summary PLOTS=none;
    변수 zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
실행;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## 4단계 — 출력 데이터셋 살펴보기

`OUTPUT OUT=`에서 나온 `clusters` 데이터셋은 센서마다 배정된 `Cluster`와 `RSq_Own`(센서와 자기 군집 성분 간의 제곱 상관관계)을 담은 행 하나를 가집니다. `RSq_Own`이 높다는 것은 해당 센서가 자기 군집으로 잘 표현된다는 뜻이며 — 군집 대표를 대신해 *제외*하기에 이상적인 후보입니다. 군집별, 그다음 R제곱별로 정렬하여 각 군집에서 가장 대표적인 센서가 그룹 맨 위에 오도록 합니다.

In [4]:
처리 정렬 데이터=clusters out=clusters_ranked;
    기준 CLUSTER DESCENDING RSq_Own;
실행;

처리 인쇄 데이터=clusters_ranked 라벨;
    변수 Variable CLUSTER RSq_Own;
    라벨 Variable = "센서 채널"
          CLUSTER  = "군집"
          RSq_Own  = "자기 군집 R제곱";
실행;


  Obs          센서 채널      군집              자기 군집 R제곱
-----  -------------  ------  ---------------------
    1  ZONE1_A             1                0.96867
    2  ZONE1_B             1                 0.9189
    3  TEMP_IN             1               0.903394
    4  TEMP_OUT            1               0.889519
    5  ZONE2_A             2                0.98364
    6  ZONE2_B             2               0.946708
    7  PRESSURE_A          2               0.929356
    8  PRESSURE_B          2               0.920915
    9  ZONE2_C             2               0.892405
   10  ZONE3_A             3               0.977237
   11  VIBRATION           3                0.95916
   12  ZONE3_B             3               0.949054
   13  ZONE1_C             4                      1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## 결과 해석

분할은 다음의 정직한 한 가지 예외와 함께 라인의 물리적 구조 대부분을 복원합니다:

- **군집 1**은 `zone1_a`(R²=0.969), `zone1_b`(0.919), 그리고 입구/출구 온도 `temp_in`(0.903)과 `temp_out`(0.890)을 모아, 구역 1 잠재 신호에 의해 구동되는 다섯 채널 중 네 개를 포함합니다. 평균 R제곱 0.920.
- **군집 2**는 세 구역-2 채널 `zone2_a`(0.984), `zone2_b`(0.947), `zone2_c`(0.892)와, 구역-2 구동 신호에 연결한 두 압력 변환기 `pressure_a`(0.929), `pressure_b`(0.921)를 함께 모읍니다. 평균 R제곱 0.935.
- **군집 3**은 `zone3_a`(0.977), `zone3_b`(0.949), `vibration`(0.959)을 모읍니다. 평균 R제곱 0.962.
- **군집 4**는 단일 채널입니다: `zone1_c`, 세 번째 구역-1 프로브가 R²=1.000으로 독립적으로 분리되었습니다(단일 채널은 자명하게 자기 자신에 의해 완벽히 설명됩니다). 구역-1 구동 신호를 공유함에도, 이 표본 크기에서 절차는 `zone1_c`가 `zone1_a`/온도 그룹과 충분히 구별된다고 판단하여 군집 1에 합치는 대신 독자적인 군집을 부여했습니다.

세 개의 다중 채널 군집은 각각 **0.90**을 넘는 평균 R제곱을 가지며, 단일 성분이 구성원 간 변동의 대부분을 설명함을 확인해줍니다 — 이것이 바로 우리가 축소하고자 하는 중복성입니다. 이상치인 `zone1_c`가 나머지 구역 1과 합쳐지지 않고 자기만의 군집에 속한 것은, 변수 군집화가 데이터 주도적임을 상기시켜줍니다: 더 많은 사이클이나 더 높은 `MAXCLUSTERS` 예산이 있으면 경계가 바뀔 수 있으므로, 이 분할은 고정된 진리가 아니라 엔지니어링 판단의 출발점입니다.

**라인에 대한 조치.** 각 다중 채널 군집 내에서는 `RSq_Own`이 가장 높은 센서(자기 군집을 가장 잘 대표하는 채널)를 유지하고 나머지는 일상 SPC 차트 작성에서 제외하거나 우선순위를 낮춥니다 — 예를 들어 군집 2는 `zone2_a`, 군집 3은 `zone3_a`. `zone1_c` 단일 채널은 자동으로 유지할 대상이 아니라 조사할 대상으로 취급합니다: 별도로 모니터링하기로 결정하기 전에 그것이 진정으로 구별되는 정보를 담고 있는지, 아니면 군집화의 부산물인지 확인하십시오. 이 한 채널을 별도로 두더라도, 이는 13개 모니터링 채널을 4채널 모니터링 계획으로 축소하여 경보 잡음과 다중공선성을 줄이면서도 라인의 정보 내용을 보존합니다.